# mDPR Baseline - GPU Accelerated (FAST!)

**Speed:** ~2-3 minutes (vs 10-15 minutes CPU)

**How:** Manually encode queries on GPU in batches

**Requirements:** T4 GPU enabled in Colab

In [2]:
# Install Java 21 and dependencies
!apt-get install -qq openjdk-21-jdk-headless
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch

print("✓ Installation complete")
print("⚠️ IMPORTANT: Click 'Runtime' → 'Restart runtime' now!")
print("⚠️ Then run the cells starting from cell 2")

Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.9+10-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.9+10-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.9+10-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.9+10-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.9+10-1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/java to provide /usr/bin/java (java) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/jpackage to provide /usr/bin/jpackage (jpackage) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/keytool to provide /usr/bin/keytool (keytool) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/b

In [1]:
# Set Java environment (run after restart)
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Verify Java version
!java -version
print("\n✓ Java 21 configured")

openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)

✓ Java 21 configured


In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from pyserini.search import get_topics, get_qrels
from pyserini.search.faiss import FaissSearcher
import pytrec_eval
from tqdm.notebook import tqdm

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Enable T4 GPU in Runtime settings")

Using device: cuda
GPU: Tesla T4
GPU Memory: 15.8 GB


In [3]:
# Load mDPR encoder on GPU
print("Loading mDPR encoder on GPU...")
model_name = 'castorini/mdpr-tied-pft-msmarco'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print("✓ Encoder loaded on GPU")

Loading mDPR encoder on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/407 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

✓ Encoder loaded on GPU


In [4]:
# Load MIRACL data
print("Loading MIRACL data...")
topics = get_topics('miracl-v1.0-ar-dev')
qrels = get_qrels('miracl-v1.0-ar-dev')
print(f"✓ Loaded {len(topics)} queries")

Loading MIRACL data...
✓ Loaded 2896 queries


In [5]:
# Load FAISS index
print("Loading FAISS index...")
print("⚠️ First run: downloading ~6GB (5-10 minutes)")

temp_searcher = FaissSearcher.from_prebuilt_index(
    'miracl-v1.0-ar-mdpr-tied-pft-msmarco',
    'castorini/mdpr-tied-pft-msmarco'
)

index = temp_searcher.index
docid_map = temp_searcher.docids
print(f"✓ Index loaded: {index.ntotal:,} documents")

Loading FAISS index...
⚠️ First run: downloading ~6GB (5-10 minutes)
Attempting to initialize prebuilt index miracl-v1.0-ar-mdpr-tied-pft-msmarco.


faiss.miracl-v1.0-ar.mdpr-tied-pft-msmarco.20221004.2b2856.tar.gz: 100%|██████████| 5.47G/5.47G [06:27<00:00, 15.2MB/s]


Extracting /root/.cache/pyserini/indexes/faiss.miracl-v1.0-ar.mdpr-tied-pft-msmarco.20221004.2b2856.tar.gz into /root/.cache/pyserini/indexes/faiss.miracl-v1.0-ar.mdpr-tied-pft-msmarco.20221004.2b2856.177d47e9a802c87abca52380ad1ce83b...
Initializing miracl-v1.0-ar-mdpr-tied-pft-msmarco...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.


lucene-index.miracl-v1.0-ar.20221004.2b2856.tar.gz: 100%|██████████| 1.11G/1.11G [02:17<00:00, 8.62MB/s]


✓ Index loaded: 2,061,414 documents


In [6]:
# GPU encoding function
@torch.no_grad()
def encode_queries_gpu(queries, batch_size=64):
    """Encode queries using GPU - THIS IS THE SPEEDUP!"""
    all_embeddings = []

    for i in tqdm(range(0, len(queries), batch_size), desc="Encoding on GPU"):
        batch = queries[i:i+batch_size]

        # Tokenize and move to GPU
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        # Encode on GPU
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token

        # Normalize
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

print("✓ GPU encoding function ready")

✓ GPU encoding function ready


In [7]:
# Encode all queries on GPU (THIS IS FAST!)
print(f"Encoding {len(topics)} queries on GPU...")
print("This will take ~1-2 minutes with T4 GPU")

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

query_embeddings = encode_queries_gpu(query_texts, batch_size=64)
print(f"✓ Encoded {len(query_embeddings)} queries")

Encoding 2896 queries on GPU...
This will take ~1-2 minutes with T4 GPU


Encoding on GPU:   0%|          | 0/46 [00:00<?, ?it/s]

✓ Encoded 2896 queries


In [8]:
# Search FAISS index (fast!)
print("Searching FAISS index...")
k = 100
scores, indices = index.search(query_embeddings.astype('float32'), k)
print("✓ Search complete")

Searching FAISS index...
✓ Search complete


In [9]:
# Format results
print("Formatting results...")
results = {}

for i, qid in enumerate(tqdm(query_ids, desc="Processing")):
    results[str(qid)] = {}
    for idx, score in zip(indices[i], scores[i]):
        if idx != -1:
            docid = docid_map[idx]
            results[str(qid)][docid] = float(score)

print(f"✓ Formatted {len(results)} query results")

Formatting results...


Processing:   0%|          | 0/2896 [00:00<?, ?it/s]

✓ Formatted 2896 query results


In [11]:
# Evaluate
print("Evaluating...")

qrels_str = {
    str(qid): {str(docid): int(rel) for docid, rel in docs.items()}
    for qid, docs in qrels.items()
}

metrics = {'recall_10', 'recall_100', 'ndcg_cut_10', 'recip_rank'}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_str, metrics)
eval_results = evaluator.evaluate(results)

# Aggregate
num_queries = len(eval_results)
aggregated = {metric: 0.0 for metric in metrics}

for qid in eval_results:
    for metric in metrics:
        aggregated[metric] += eval_results[qid][metric]

avg_metrics = {metric: score / num_queries for metric, score in aggregated.items()}

# Print results
print("\n" + "="*60)
print("mDPR BASELINE RESULTS - GPU Accelerated")
print("="*60)
print(f"Recall@10:  {avg_metrics['recall_10']:.4f}")
print(f"Recall@100: {avg_metrics['recall_100']:.4f} (Expected: ~0.841)")
print(f"NDCG@10:    {avg_metrics['ndcg_cut_10']:.4f} (Expected: ~0.499)")
print(f"MRR:        {avg_metrics['recip_rank']:.4f}")
print(f"Queries:    {num_queries}")
print("="*60)

Evaluating...

mDPR BASELINE RESULTS - GPU Accelerated
Recall@10:  0.6156
Recall@100: 0.8407 (Expected: ~0.841)
NDCG@10:    0.4993 (Expected: ~0.499)
MRR:        0.5328
Queries:    2896


In [ ]:
# Save results
import os
os.makedirs('results', exist_ok=True)

output_file = 'results/mdpr_baseline_gpu.txt'
with open(output_file, 'w') as f:
    for qid, docs in results.items():
        sorted_docs = sorted(docs.items(), key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(sorted_docs, 1):
            f.write(f"{qid} Q0 {docid} {rank} {score:.6f} mdpr_gpu\n")

print(f"✓ Results saved to {output_file}")